In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("darkgrid")
import datetime

from kiblib.utils.db import DbConn

In [2]:
db_conn = DbConn().create_engine()

In [166]:
today = datetime.datetime.now()

#dates = [datetime.date(2023, d, 1).strftime("%Y-%m-%d") for d in range(1, 13)]
#dates.extend([datetime.date(2024, d, 1).strftime("%Y-%m-%d") for d in range(1, 13)])
#dates.extend([datetime.date(2025, d, 1).strftime("%Y-%m-%d") for d in range(1, 7)])
#dates

# tous les mardis entre deux dates extrêmes
start, end = datetime.datetime(2024, 1, 1), datetime.datetime(2025, 1, 1)
days = (start + datetime.timedelta(days=i) for i in range((end - start).days + 1))
dates = [d.strftime("%Y-%m-%d") for d in days if d.weekday() == 1 ]

In [158]:
prets_all = pd.DataFrame()
for date in dates:
    print(date)
    query = f"SELECT * FROM statdb.stat_issues WHERE DATE(issuedate) <= '{date}' AND DATE(date_due) > '{date}' AND ( DATE(returndate) > '{date}' OR returndate IS NULL)"
    prets = pd.read_sql(query, con=db_conn)
    prets['date_pivot'] = date
    prets = prets[prets['categorycode'].isin(['BIBL', 'MEDA', 'MEDB', 'MEDC', 'CSVT'])]

    prets['issuedate_'] = pd.to_datetime(prets['issuedate'])
    prets['date_due_'] = pd.to_datetime(prets['date_due'])
    prets['returndate_'] = pd.to_datetime(prets['returndate'])

    prets.loc[~prets['returndate'].isna(), 'retard'] = round( (prets['returndate_'] - prets['date_due_']) / np.timedelta64(1, 'D'), 1 )
    prets.loc[prets['returndate'].isna(), 'retard'] = round( (today - prets['date_due_']) / np.timedelta64(1, 'D'), 1 )

    prets['retard?'] = np.nan
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] <= 0), 'retard?'] = '0 - pas de retard'
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] > 180), 'retard?'] = '7 - contentieux 180 jours'
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] > 90), 'retard?'] = '6 - contentieux 90 jours'
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] > 42), 'retard?'] = '5 - contentieux'
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] > 28), 'retard?'] = '4 - retard > 28 jours'
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] > 14), 'retard?'] = '3 - retard > 14 jours'
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] > 7), 'retard?'] = '2 - retard > 7 jours'
    prets.loc[(prets['retard?'].isna()) & (prets['retard'] > 0), 'retard?'] = '1 - retard <= 7 jours'    

    prets_all = pd.concat([prets_all, prets])

2024-01-02
2024-01-09
2024-01-16
2024-01-23
2024-01-30
2024-02-06
2024-02-13
2024-02-20
2024-02-27
2024-03-05
2024-03-12
2024-03-19
2024-03-26
2024-04-02
2024-04-09
2024-04-16
2024-04-23
2024-04-30
2024-05-07
2024-05-14
2024-05-21
2024-05-28
2024-06-04
2024-06-11
2024-06-18
2024-06-25
2024-07-02
2024-07-09
2024-07-16
2024-07-23
2024-07-30
2024-08-06
2024-08-13
2024-08-20
2024-08-27
2024-09-03
2024-09-10
2024-09-17
2024-09-24
2024-10-01
2024-10-08
2024-10-15
2024-10-22
2024-10-29
2024-11-05
2024-11-12
2024-11-19
2024-11-26
2024-12-03
2024-12-10
2024-12-17
2024-12-24
2024-12-31
2025-01-07
2025-01-14
2025-01-21
2025-01-28
2025-02-04
2025-02-11
2025-02-18
2025-02-25
2025-03-04
2025-03-11
2025-03-18
2025-03-25
2025-04-01
2025-04-08
2025-04-15
2025-04-22


KeyboardInterrupt: 

In [168]:
prets_all = prets_all[~prets_all['date_pivot'].isin(['2025-01-07', '2025-01-14', '2025-01-21', 
                                                     '2025-01-28', '2025-02-04', '2025-02-11',
                                                     '2025-02-18', '2025-02-25', '2025-03-04',
                                                     '2025-03-11', '2025-03-18', '2025-03-25',
                                                     '2025-04-01', '2025-04-08', '2025-04-15'])]

In [169]:
prets_all['date_pivot'].unique()

array(['2024-01-02', '2024-01-09', '2024-01-16', '2024-01-23',
       '2024-01-30', '2024-02-06', '2024-02-13', '2024-02-20',
       '2024-02-27', '2024-03-05', '2024-03-12', '2024-03-19',
       '2024-03-26', '2024-04-02', '2024-04-09', '2024-04-16',
       '2024-04-23', '2024-04-30', '2024-05-07', '2024-05-14',
       '2024-05-21', '2024-05-28', '2024-06-04', '2024-06-11',
       '2024-06-18', '2024-06-25', '2024-07-02', '2024-07-09',
       '2024-07-16', '2024-07-23', '2024-07-30', '2024-08-06',
       '2024-08-13', '2024-08-20', '2024-08-27', '2024-09-03',
       '2024-09-10', '2024-09-17', '2024-09-24', '2024-10-01',
       '2024-10-08', '2024-10-15', '2024-10-22', '2024-10-29',
       '2024-11-05', '2024-11-12', '2024-11-19', '2024-11-26',
       '2024-12-03', '2024-12-10', '2024-12-17', '2024-12-24',
       '2024-12-31'], dtype=object)

In [163]:
table = prets_all.pivot_table(index='date_pivot', columns=['retard?'], values=['issue_id'],
                              aggfunc={'issue_id': "count"},
                              margins=True,
                              margins_name= 'Total'
                             )
table

issue_id                                             \
retard?    0 - pas de retard 1 - retard <= 7 jours 2 - retard > 7 jours   
date_pivot                                                                
2024-01-02              9404                  2766                 1401   
2024-01-09              9327                  2733                 1452   
2024-01-16              9875                  2608                 1291   
2024-01-23             10315                  2317                 1022   
2024-01-30             10599                  2396                 1151   
2024-02-06             11106                  2413                 1269   
2024-02-13             11041                  2494                 1427   
2024-02-20             10922                  2684                 1381   
2024-02-27             10601                  2855                 1518   
2024-03-05             10430                  3029                 1601   
2024-03-12             10232                  2627                 1479   
2024-03-19             10298                  2739                 1422   
2024-03-26             10534                  2759                 1413   
2024-04-02             10413                  2910                 1486   
2024-04-09              9894                  2530                 1306   
2024-04-16              9671                  2225                 1266   
2024-04-23              9229                  2752                 1320   
2024-04-30              9771                  2784                 1621   
2024-05-07              9196                  2951                 1608   
2024-05-14              9049                  2056                 1335   
2024-05-21              9468                  2029                 1086   
2024-05-28              9164                  2028                 1106   
2024-06-04              8772                  2123                 1283   
2024-06-11              8245                  2570                 1362   
2024-06-18              8333                  2298                 1095   
2024-06-25              8559                  2158                  965   
2024-07-02              8927                  1762                 1023   
2024-07-09              8722                  2018                 1429   
2024-07-16              9019                  2386                 1652   
2024-07-23              8672                  2344                 1496   
2024-07-30              8424                  2325                 1360   
2024-08-06              7979                  2374                 1300   
2024-08-13              7961                  2514                 1277   
2024-08-20              8355                  2319                 1207   
2024-08-27              8621                  2146                 1217   
2024-09-03              8771                  2209                 1157   
2024-09-10              9550                  2465                 1101   
2024-09-17              9865                  2421                 1126   
2024-09-24             10090                  2409                 1289   
2024-10-01             10121                  2487                 1482   
2024-10-08             10524                  2526                 1530   
2024-10-15             10479                  2535                 1582   
2024-10-22             10544                  2907                 1609   
2024-10-29             10494                  2892                 1624   
2024-11-05             10367                  2661                 1408   
2024-11-12             10619                  2251                 1289   
2024-11-19             10838                  2462                 1365   
2024-11-26             11033                  2502                 1268   
2024-12-03             10529                  2231                 1352   
2024-12-10              9984                  2103                 1328   
2024-12-17              9127                 

In [164]:
table2 =table.div(table.iloc[:,-1], axis=0 )
table2

issue_id                                             \
retard?    0 - pas de retard 1 - retard <= 7 jours 2 - retard > 7 jours   
date_pivot                                                                
2024-01-02          0.540522              0.158984             0.080526   
2024-01-09          0.542141              0.158858             0.084399   
2024-01-16          0.575198              0.151911             0.075198   
2024-01-23          0.615747              0.138312             0.061008   
2024-01-30          0.611105              0.138146             0.066363   
2024-02-06          0.602735              0.130956             0.068870   
2024-02-13          0.590807              0.133455             0.076359   
2024-02-20          0.581267              0.142842             0.073497   
2024-02-27          0.563493              0.151757             0.080689   
2024-03-05          0.547823              0.159094             0.084091   
2024-03-12          0.565273              0.145130             0.081708   
2024-03-19          0.566353              0.150635             0.078205   
2024-03-26          0.578029              0.151394             0.077535   
2024-04-02          0.565524              0.158041             0.080704   
2024-04-09          0.574765              0.146973             0.075868   
2024-04-16          0.581225              0.133722             0.076086   
2024-04-23          0.541672              0.161521             0.077474   
2024-04-30          0.537429              0.153127             0.089159   
2024-05-07          0.516078              0.165610             0.090241   
2024-05-14          0.564398              0.128236             0.083266   
2024-05-21          0.586690              0.125728             0.067295   
2024-05-28          0.561451              0.124249             0.067761   
2024-06-04          0.529678              0.128193             0.077471   
2024-06-11          0.502805              0.156726             0.083059   
2024-06-18          0.553357              0.152600             0.072714   
2024-06-25          0.579289              0.146058             0.065313   
2024-07-02          0.607941              0.119995             0.069668   
2024-07-09          0.552340              0.127794             0.090495   
2024-07-16          0.533574              0.141158             0.097734   
2024-07-23          0.531927              0.143777             0.091762   
2024-07-30          0.535027              0.147666             0.086377   
2024-08-06          0.526458              0.156638             0.085775   
2024-08-13          0.531904              0.167970             0.085321   
2024-08-20          0.569336              0.158024             0.082249   
2024-08-27          0.596816              0.148564             0.084251   
2024-09-03          0.587554              0.147977             0.077505   
2024-09-10          0.586681              0.151431             0.067637   
2024-09-17          0.586016              0.143816             0.066888   
2024-09-24          0.589163              0.140663             0.075266   
2024-10-01          0.575220              0.141347             0.084228   
2024-10-08          0.581951              0.139681             0.084605   
2024-10-15          0.577737              0.139762             0.087220   
2024-10-22          0.568410              0.156712             0.086739   
2024-10-29          0.558251              0.153846             0.086392   
2024-11-05          0.570117              0.146337             0.077431   
2024-11-12          0.585037              0.124015             0.071015   
2024-11-19          0.578088              0.131321             0.072808   
2024-11-26          0.575025              0.130401             0.066086   
2024-12-03          0.567693              0.120289             0.072896   
2024-12-10          0.552916              0.116465             0.073545   
2024-12-17          0.516525              0.1

In [170]:
table2.to_excel("test.xlsx")

In [172]:
p = prets_all[prets_all['retard?'] == '7 - contentieux 180 jours']
p2 = p[p['date_pivot'] == '2024-12-17']

In [173]:
p2

,issuedate,date_due,returndate,renewals,branch,arret_bus,borrowernumber,cardnumber,age,sexe,...,cle,timestamp,issue_id,returnbranch,date_pivot,issuedate_,date_due_,returndate_,retard,retard?
10200,2024-11-06 11:05:10,2024-12-18 23:59:00,NaT,1,MED,None,23468,X0002491691,51,F,...,2024-11-06 11:05:10-466878,2024-11-08 03:12:06,5112087,None,2024-12-17,2024-11-06 11:05:10,2024-12-18 23:59:00,NaT,217.7,7 - contentieux 180 jours
10219,2024-11-06 12:25:59,2024-12-18 23:59:00,NaT,1,MED,None,76243,X0002734514,12,F,...,2024-11-06 12:25:59-421475,2024-11-08 03:12:06,5112419,None,2024-12-17,2024-11-06 12:25:59,2024-12-18 23:59:00,NaT,217.7,7 - contentieux 180 jours
10220,2024-11-06 12:26:04,2024-12-18 23:59:00,NaT,1,MED,None,76243,X0002734514,12,F,...,2024-11-06 12:26:04-401945,2024-11-08 03:12:06,5112420,None,2024-12-17,2024-11-06 12:26:04,2024-12-18 23:59:00,NaT,217.7,7 - contentieux 180 jours
10222,2024-11-06 12:30:23,2024-12-18 23:59:00,NaT,1,MED,None,76243,X0002734514,12,F,...,2024-11-06 12:30:23-465628,2024-11-08 03:12:06,5112424,None,2024-12-17,2024-11-06 12:30:23,2024-12-18 23:59:00,NaT,217.7,7 - contentieux 180 jours
10223,2024-11-06 12:30:26,2024-12-18 23:59:00,NaT,1,MED,None,76243,X0002734514,12,F,...,2024-11-06 12:30:26-475646,2024-11-08 03:12:06,5112425,None,2024-12-17,2024-11-06 12:30:26,2024-12-18 23:59:00,NaT,217.7,7 - contentieux 180 jours
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30352,2024-12-17 18:02:49,2025-01-07 23:59:00,NaT,0,BUS,B03,73890,X0002752785,11,F,...,2024-12-17 18:02:49-347590,2024-12-19 03:12:28,5171817,None,2024-12-17,2024-12-17 18:02:49,2025-01-07 23:59:00,NaT,197.7,7 - contentieux 180 jours
30353,2024-12-17 18:02:53,2025-01-07 23:59:00,NaT,0,BUS,B03,73890,X0002752785,11,F,...,2024-12-17 18:02:53-479698,2024-12-19 03:12:28,5171818,None,2024-12-17,2024-12-17 18:02:53,2025-01-07 23:59:00,NaT,197.7,7 - contentieux 180 jours
30354,2024-12-17 18:03:15,2025-01-07 23:59:00,NaT,0,BUS,B03,79364,X0002611921,7,F,...,2024-12-17 18:03:15-314081,2024-12-19 03:12:28,5171819,None,2024-12-17,2024-12-17 18:03:15,2025-01-07 23:59:00,NaT,197.7,7 - contentieux 180 jours
30355,2024-12-17 18:03:20,2025-01-07 23:59:00,NaT,0,BUS,B03,79364,X0002611921,7,F,...,2024-12-17 18:03:20-384218,2024-12-19 03:12:28,5171820,None,2024-12-17,2024-12-17 18:03:20,2025-01-07 23:59:00,NaT,197.7,7 - contentieux 180 jours
